## 1. Uvoz biblioteka

In [40]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json
import time
from sentence_transformers import SentenceTransformer
import umap
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

## 2. Učitavanje i čišćenje podataka

In [41]:
dataset = load_dataset("booksouls/goodreads-book-descriptions")
df = dataset["train"].to_pandas()

df["desc_len"] = df["description"].fillna("").str.len()

# Filtriramo prekratke (nedovoljno semantičkog sadržaja) i predugačke (verovatno scraping greške) opise
df_clean = df[(df["desc_len"] >= 50) & (df["desc_len"] <= 3000)].copy()

print(f"Pre filtriranja: {len(df)}")
print(f"Posle filtriranja: {len(df_clean)}")

Pre filtriranja: 1021106
Posle filtriranja: 1007319


## 3. Uzorkovanje

In [42]:
SAMPLE_SIZE = 50000

df_sample = df_clean.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Uzorak: {len(df_sample)} knjiga")
df_sample.head()

Uzorak: 50000 knjiga


,title,description,desc_len
0,Urinary Tract Infection: Urinary Tract Infecti...,Urinary Tract Infection Treatment Guide To Cur...,2601
1,Lost Found: The Adoption Experience,"Rich in insight and compassion, Lost and Found...",1023
2,"The Cat, the Quilt and the Corpse: A Cats in T...","the author of the yellow rose mystery series, ...",720
3,"Wynken, Blynken, Nod","""This classic bedtime poem appears in a newly ...",375
4,"Hardscrabble Road (Gregor Demarkian, #21)",When Philadelphia's right-wing-ranting radio h...,782


## 4. Embeddinzi opisa

Ovaj korak najduže traje. Rezultat se čuva na disk odmah nakon, da ga ne računamo ponovo ako nastavljamo kasnije.

In [43]:
model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")

descriptions = df_sample["description"].tolist()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### 4a. (Opciono) Procena brzine pre punog enkodovanja

Meri brzinu na malom uzorku i ekstrapolira na ceo `SAMPLE_SIZE` — korisno da proceniš koliko će trajati pre nego što pokreneš enkodovanje svih opisa.

In [ ]:
subset = descriptions[:5000]
start = time.time()
_ = model.encode(subset, batch_size=32, show_progress_bar=False, device="cuda")
elapsed = time.time() - start

rate = len(subset) / elapsed
print(f"Brzina: {rate:.1f} opisa/sek")
print(f"Procena za {SAMPLE_SIZE:,}: {SAMPLE_SIZE/rate/60:.1f} minuta")

In [44]:
embeddings = model.encode(
    descriptions,
    batch_size=32,
    show_progress_bar=True,
    device="cuda",
    normalize_embeddings=True  # bge modeli su trenirani za kosinusnu sličnost preko normalizovanih vektora
)

print(f"Oblik embeddinga: {embeddings.shape}")  # (SAMPLE_SIZE, 1024) - bge-large ima 1024-dim izlaz

np.save("book_embeddings_bge.npy", embeddings)   # <- nov naziv fajla, da ne prepišemo stare (mpnet) embeddinge
df_sample.to_pickle("book_sample.pkl")

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Oblik embeddinga: (50000, 1024)


### 4b. (Opciono) Učitavanje već sačuvanih embeddinga

Pokreni OVU ćeliju umesto gornje ako nastavljaš rad u novoj sesiji i već imaš `book_embeddings_bge.npy` / `book_sample.pkl` sačuvane sa diska — preskačeš ponovno (sporo) računanje embeddinga.

In [ ]:
# embeddings = np.load("book_embeddings_bge.npy")
# df_sample = pd.read_pickle("book_sample.pkl")

## 5. Skraćen opis za prikaz u UI-ju

In [45]:
def truncate_description(text, max_len=200):
    if len(text) <= max_len:
        return text
    return text[:max_len].rsplit(" ", 1)[0] + "..."

df_sample["description_short"] = df_sample["description"].apply(truncate_description)

## 6. UMAP — redukcija na 3D za vizuelizaciju

In [46]:
reducer_3d = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=3,
    random_state=42
)

embedding_3d = reducer_3d.fit_transform(embeddings)  # radi nad originalnim (1024D) embeddinzima, ne redukovanim

print(f"Oblik 3D projekcije: {embedding_3d.shape}")

C:\Users\dimit\OneDrive\Desktop\AI PROJEKAT\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Oblik 3D projekcije: (50000, 3)


## 7. K-means klasterovanje

Radi se na originalnim embeddinzima, NE na UMAP projekciji.

### 7a. (Opciono) Provera optimalnog broja klastera

Silhouette score za tekstualne embeddinge je uvek nizak (curse of dimensionality) — ne traži se "visoka" vrednost, nego tačka gde prestaje da značajno raste (elbow). Već pokrenuto za ovaj dataset: **K=35** je izabran kao razuman kompromis (dovoljno fine podele za 100k knjiga, čitljiva legenda u UI-ju).

In [ ]:
for k in [27, 35, 45, 55, 65]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels, sample_size=5000, random_state=42)
    print(f"K={k}: silhouette={score:.3f}")

### 7b. Finalno klasterovanje

In [47]:
N_CLUSTERS = 35

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

unique, counts = np.unique(cluster_labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"Klaster {cluster_id}: {count} knjiga")

Klaster 0: 1836 knjiga
Klaster 1: 1299 knjiga
Klaster 2: 2214 knjiga
Klaster 3: 1042 knjiga
Klaster 4: 829 knjiga
Klaster 5: 1348 knjiga
Klaster 6: 1202 knjiga
Klaster 7: 1825 knjiga
Klaster 8: 1166 knjiga
Klaster 9: 1790 knjiga
Klaster 10: 923 knjiga
Klaster 11: 1113 knjiga
Klaster 12: 1479 knjiga
Klaster 13: 1316 knjiga
Klaster 14: 1568 knjiga
Klaster 15: 1260 knjiga
Klaster 16: 643 knjiga
Klaster 17: 790 knjiga
Klaster 18: 1176 knjiga
Klaster 19: 1648 knjiga
Klaster 20: 1441 knjiga
Klaster 21: 998 knjiga
Klaster 22: 1739 knjiga
Klaster 23: 1683 knjiga
Klaster 24: 1019 knjiga
Klaster 25: 1773 knjiga
Klaster 26: 2103 knjiga
Klaster 27: 1170 knjiga
Klaster 28: 2075 knjiga
Klaster 29: 1284 knjiga
Klaster 30: 1806 knjiga
Klaster 31: 1424 knjiga
Klaster 32: 1471 knjiga
Klaster 33: 1603 knjiga
Klaster 34: 1944 knjiga


## 8. Žanr — zero-shot klasifikacija

**Napomena:** ova lista žanrova je ručno sastavljena (ne dolazi iz podataka) — model bira najbliži žanr SA OVE liste za svaku knjigu, preko poređenja embeddinga.

In [48]:
genre_labels = [
    "romance novel", "fantasy novel", "science fiction", "mystery and thriller",
    "horror", "historical fiction", "biography and memoir", "self-help",
    "cooking and food", "religion and spirituality", "poetry",
    "comics and graphic novels", "children's book", "young adult fiction",
    "science and technology", "business and economics", "travel",
    "history", "politics", "psychology", "art and design", "sports", "humor"
]

genre_embeddings = model.encode(genre_labels, normalize_embeddings=True)
similarity_matrix = cosine_similarity(embeddings, genre_embeddings)
best_genre_idx = similarity_matrix.argmax(axis=1)

df_sample["genre"] = [genre_labels[i] for i in best_genre_idx]
df_sample["genre_confidence"] = similarity_matrix.max(axis=1)

print(df_sample["genre"].value_counts())

genre
romance novel                7299
mystery and thriller         6166
fantasy novel                5158
young adult fiction          4172
biography and memoir         4122
children's book              3511
historical fiction           3300
religion and spirituality    2354
science fiction              1912
self-help                    1895
comics and graphic novels    1545
poetry                       1405
horror                       1157
art and design                970
business and economics        920
cooking and food              897
psychology                    665
science and technology        646
humor                         526
politics                      500
travel                        367
sports                        289
history                       224
Name: count, dtype: int64


### 8a. (Opciono) Provera pokrivenosti liste žanrova

Nizak prosečan `genre_confidence` za neki žanr može značiti da mu ta kategorija ne odgovara najbolje (guranje u "najbliži dostupan" žanr jer prava kategorija ne postoji na listi) — vredi ručno pogledati par primera pre nego što dodaš/menjaš kategorije.

In [ ]:
print(df_sample.groupby("genre")["genre_confidence"].mean().sort_values())

In [ ]:
# Ručna provera konkretnih primera za žanr(ove) sa najnižim confidence-om
for genre in ["history", "travel"]:
    sample = df_sample[df_sample["genre"] == genre].sample(5, random_state=1)
    for _, row in sample.iterrows():
        print(f"[{row['genre_confidence']:.3f}] {row['title']}")
        print(f"  {row['description_short'][:150]}")

## 9. Prava top-K sličnost (za "najsličnije knjige" u UI-ju)

Računa se na originalnim embeddinzima (ne UMAP koordinatama) — ovo je matematički tačna sličnost, koju frontend čita direktno iz JSON-a bez ikakvog računanja u browseru.

Radi se u batch-ovima (ne kao jedna puna N×N matrica) da ne eksplodira RAM na velikim dataset-ima — puna matrica za 100k knjiga bi zauzela ~40GB.

In [49]:
K = 8  # čuvamo malo više od 5, kao rezervu za filter po žanru na frontend-u
BATCH_SIZE = 1000  # smanji na 500 ako i dalje puca RAM

n = len(embeddings)
similar_indices = np.zeros((n, K), dtype=np.int32)
top1_similarity = np.zeros(n, dtype=np.float32)  # čuvamo i skor najbližeg komšije, koristan za evaluaciju kasnije

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch_sim = cosine_similarity(embeddings[start:end], embeddings)  # (batch, n) - ne (n, n)

    # isključujemo sličnost knjige sa samom sobom
    for i, row_idx in enumerate(range(start, end)):
        batch_sim[i, row_idx] = -1

    # argpartition je brži od punog argsort-a kad ti treba samo top-K
    top_k_unsorted = np.argpartition(-batch_sim, K, axis=1)[:, :K]
    for i in range(end - start):
        row = top_k_unsorted[i]
        order = row[np.argsort(-batch_sim[i, row])]  # sortiramo tih K po stvarnoj sličnosti
        similar_indices[start + i] = order
        top1_similarity[start + i] = batch_sim[i, order[0]]

    print(f"Obrađeno {end}/{n}", end="\r")

df_sample["similar_ids"] = similar_indices.tolist()
print()

Obrađeno 50000/50000


## 10. Finalni izvoz u JSON (3D verzija)

In [50]:
df_sample["x"] = embedding_3d[:, 0]
df_sample["y"] = embedding_3d[:, 1]
df_sample["z"] = embedding_3d[:, 2]
df_sample["cluster"] = cluster_labels

output_data = df_sample[[
    "title", "description_short", "x", "y", "z", "cluster", "genre", "similar_ids"
]].to_dict(orient="records")

with open("books_data_3d.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Sačuvano {len(output_data)} knjiga u books_data_3d.json")

Sačuvano 50000 knjiga u books_data_3d.json


In [51]:
# Sanity check - dužine svih ključnih struktura moraju da se poklapaju
print(f"df_sample: {len(df_sample)}")
print(f"embeddings: {embeddings.shape[0]}")
print(f"embedding_3d: {embedding_3d.shape[0]}")

df_sample: 50000
embeddings: 50000
embedding_3d: 50000


## 11. (Opciono) Evaluacija

Provera da li je embedding sličnost stvarno smislena, ne samo vizuelno lepa. Silhouette score meri matematičku kompaktnost klastera, NE da li podela ima smisla za čoveka (niske vrednosti su uobičajene za tekstualne embeddinge) — zato koristimo i druge, konkretnije provere ispod.

### 11a. Silhouette score za finalni broj klastera

In [ ]:
score = silhouette_score(embeddings, cluster_labels, sample_size=5000, random_state=42)
print(f"Silhouette score (K={N_CLUSTERS}): {score:.3f}")

### 11b. Manuelni spot-check — top-5 najsličnijih za konkretne naslove

Jedna funkcija za ponovnu upotrebu: prosleđuješ listu naslova (birani ručno ili nasumično) i gledaš da li rangovi imaju smisla za čoveka.

In [ ]:
def print_top_similar(titles, k=5):
    for target_title in titles:
        idx_match = df_sample[df_sample["title"] == target_title].index
        if len(idx_match) == 0:
            print(f"'{target_title}' nije u uzorku.\n")
            continue
        target_idx = idx_match[0]
        sims = cosine_similarity(embeddings[target_idx:target_idx+1], embeddings)[0]
        sims[target_idx] = -1
        top_k = sims.argsort()[-k:][::-1]
        print(f"--- {target_title} ---")
        for i in top_k:
            print(f"[{sims[i]:.3f}] {df_sample.loc[i, 'title']}")
        print()

In [ ]:
# Konkretni, ranije provereni primeri
print_top_similar(["Devon's Last Day", "Gone with the Wind, Part 2"])

In [ ]:
# Nasumičan uzorak naslova, da izbegnemo pristrasnost u ručnom biranju primera
np.random.seed(1)  # ukloni ako želiš svaki put drugačije naslove
sample_titles = df_sample["title"].sample(8, random_state=1).tolist()
print_top_similar(sample_titles)

### 11c. Genre agreement@K — kvantitativna provera

Koliki procenat top-K "najsličnijih" knjiga zaista deli isti žanr sa originalnom knjigom.

In [ ]:
def genre_agreement_at_k(df, k=5):
    hits = []
    for idx, row in df.iterrows():
        neighbor_ids = row["similar_ids"][:k]
        same_genre = sum(df.loc[n, "genre"] == row["genre"] for n in neighbor_ids)
        hits.append(same_genre / k)
    return np.mean(hits)

print(f"Genre agreement@5: {genre_agreement_at_k(df_sample):.1%}")

### 11d. Baseline poređenje — model naspram slučajnosti

Dva nezavisna testa protiv nasumičnog izbora: (1) da li je sličnost sa najbližim komšijom jasno viša od sličnosti nasumičnih parova, i (2) da li je genre agreement iznad nasumičnog poklapanja žanra.

Koristi već izračunate, normalizovane embeddinge — pošto su vektori jedinične dužine, kosinusna sličnost je prost dot produkt, pa ne treba ponovo praviti veliku N×N matricu.

In [ ]:
# top1_similarity je već izračunat u sekciji 9 (skor najbližeg komšije za svaku knjigu)
random_pairs = np.random.choice(len(embeddings), (2000, 2))
random_sims = np.sum(embeddings[random_pairs[:, 0]] * embeddings[random_pairs[:, 1]], axis=1)

print(f"Random parovi: mean={np.mean(random_sims):.3f}")
print(f"Top-1 komšije: mean={np.mean(top1_similarity):.3f}")

In [ ]:
random_pairs_genre = np.random.choice(len(df_sample), (2000, 2))
random_genre_agreement = np.mean([
    df_sample.iloc[i]["genre"] == df_sample.iloc[j]["genre"]
    for i, j in random_pairs_genre
])
print(f"Random genre agreement baseline: {random_genre_agreement:.1%}")